# Deadtime explorer (heatmap)

Interactive views for the double-pulse deadtime scans.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, Checkbox, IntSlider
from IPython.display import clear_output
import sys

sns.set_context('talk')

SELECT = "(select)"

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / 'src'))
from deadtime_analysis import DeadtimeAnalysis

DATA_FILES = [
    ROOT / 'data' / 'double_pulse_deadtime-01-14-26.jsonl',
]

analysis_full = DeadtimeAnalysis.from_jsonl([str(p) for p in DATA_FILES])

est_df = None
EST_PATH = ROOT / 'data' / 'estimated_deadtime_01-14-26_packet.json'
if EST_PATH.exists():
    with EST_PATH.open() as fh:
        est_df = pd.DataFrame(json.load(fh))
print('Loaded', len(analysis_full.df), 'rows from', len(DATA_FILES), 'files')
if est_df is not None:
    print('Estimates rows:', len(est_df))


Loaded 14274 rows from 1 files
Estimates rows: 793


## Heatmap of min multi-pulse separation

In [2]:
if est_df is None:
    raise RuntimeError('No estimates loaded; cannot plot heatmap.')

pulse_rate_heatmap = [SELECT] + sorted(est_df['pulse_rate_hz'].dropna().unique())
pulse_count_heatmap = [SELECT] + sorted(est_df['num_pulses'].dropna().unique())

@interact(
    pulse_rate=Dropdown(options=pulse_rate_heatmap, value=SELECT, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_count_heatmap, value=SELECT, description='Pulses'),
    show_numbers=Checkbox(value=True, description='Show numbers'),
    number_size=IntSlider(value=8, min=4, max=18, step=1, description='Number size'),
    decimals=Dropdown(options=[0, 1, 2, 3, 4], value=2, description='Decimals'),
)
def _plot_heatmap(pulse_rate, num_pulses, show_numbers, number_size, decimals):
    clear_output(wait=True)
    if pulse_rate in (None, SELECT) or num_pulses in (None, SELECT):
        return
    sub = est_df[(est_df['pulse_rate_hz'] == pulse_rate) & (est_df['num_pulses'] == num_pulses)]
    if sub.empty:
        print('No estimates for selection')
        return
    pivot = sub.pivot_table(index='channel_count', columns='windows', values='min_all_pulses_response_us')
    fmt = f'.{decimals}f' if show_numbers else ''
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        pivot,
        annot=show_numbers,
        annot_kws={'size': number_size},
        fmt=fmt,
        cmap='viridis',
    )
    plt.title(f'Min multi-pulse separation (us, full channels) @ {pulse_rate:.0f} Hz, pulses={int(num_pulses)}')
    plt.xlabel('Windows')
    plt.ylabel('Channels')
    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=('(select)', np.float64(100.0)), value='…